# 衡东方言 ASR - 长音频测试

使用 extra_long_talks 中的 6 条长录音（10-30 分钟），评估微调后模型在真实连续语音场景下的表现。

## 1. 加载模型

使用 VAD + SenseVoice pipeline，自动处理长音频。

In [ ]:
from funasr import AutoModel
import json, os, re, torch

# 查找 SenseVoice 模型路径
SV_DIR = "/mnt/output/sensevoice_lora"
if not os.path.exists(SV_DIR):
    SV_DIR = "/mnt/data/output/sensevoice_lora"

SV_CKPT = f"{SV_DIR}/model.pt.best"
print(f"加载基座模型: iic/SenseVoiceSmall (LoRA)")
print(f"Checkpoint: {SV_CKPT}")

model = AutoModel(
    model="iic/SenseVoiceSmall", lora_only=True, disable_update=True,
    vad_model="iic/speech_fsmn_vad_zh-cn-16k-common-pytorch",
    vad_kwargs={"max_single_segment_time": 30000},
)
ckpt = torch.load(SV_CKPT, map_location="cpu")

# 检查 key 匹配
model_keys = set(model.model.state_dict().keys())
ckpt_keys = set(ckpt["state_dict"].keys())
matched = model_keys & ckpt_keys
print(f"Key 匹配: {len(matched)}/{len(model_keys)}")

model.model.load_state_dict(ckpt["state_dict"], strict=False)
print("模型加载成功!")

## 2. 加载长音频测试数据

In [ ]:
LONG_TALKS_JSONL = "/mnt/data/hengdong_asr_trainset/manifests/extra_long_talks.jsonl"

samples = []
with open(LONG_TALKS_JSONL) as f:
    for line in f:
        item = json.loads(line)
        samples.append({
            'id': item['id'],
            'audio': item['audio_filepath'].replace(
                '/Users/fanhua/Desktop/hengdong_asr_trainset', 
                '/mnt/data/hengdong_asr_trainset'
            ),
            'text': item['text'],
            'duration': item['duration'],
            'speaker': item.get('speaker_id', 'unknown'),
        })

print(f"加载 {len(samples)} 条长音频:\n")
for s in samples:
    exists = os.path.exists(s['audio'])
    print(f"  {s['speaker']} | {s['duration']/60:.1f}min | 文件{'存在' if exists else '不存在'}")

## 3. CER 计算工具

In [ ]:
try:
    import editdistance
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "editdistance"], check=True)
    import editdistance

PUNCT_PATTERN = re.compile(r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b]')

def normalize_text(text):
    return PUNCT_PATTERN.sub('', text)

def compute_cer(hyp, ref):
    hyp = normalize_text(hyp)
    ref = normalize_text(ref)
    if not ref:
        return 0.0
    return editdistance.eval(hyp, ref) / len(ref)

print("CER 工具准备就绪")

## 4. 运行长音频测试

In [ ]:
results = []

for i, s in enumerate(samples):
    audio = s['audio']
    ref_text = s['text']
    
    if not os.path.exists(audio):
        print(f"[{i+1}/{len(samples)}] 跳过，文件不存在: {audio}")
        continue
    
    print(f"[{i+1}/{len(samples)}] {s['speaker']} ({s['duration']/60:.1f}min)")
    
    result = model.generate(input=audio, language="auto", use_itn=True)
    pred_text = "".join(r['text'] for r in result if 'text' in r)
    cer = compute_cer(pred_text, ref_text)
    
    results.append({
        'id': s['id'],
        'speaker': s['speaker'],
        'audio': audio,
        'duration_min': s['duration'] / 60,
        'cer': cer,
        'ref_len': len(normalize_text(ref_text)),
        'pred_len': len(normalize_text(pred_text)),
        'expected': ref_text,
        'predicted': pred_text,
    })
    
    print(f"  CER: {cer:.1%} | 标注: {len(normalize_text(ref_text))}字 | 识别: {len(normalize_text(pred_text))}字")
    print(f"  预测前100字: {pred_text[:100]}...")
    print()

## 5. 汇总结果

In [ ]:
print("=" * 60)
print("长音频测试汇总")
print("=" * 60)
print()

if results:
    avg_cer = sum(r['cer'] for r in results) / len(results)
    
    print(f"{'说话人':<20} {'时长':>6} {'CER':>8} {'标注字数':>8} {'识别字数':>8}")
    print("-" * 55)
    for r in results:
        print(f"{r['speaker']:<20} {r['duration_min']:>5.1f}m {r['cer']:>7.1%} {r['ref_len']:>8} {r['pred_len']:>8}")
    print("-" * 55)
    print(f"{'平均':<20} {'':>6} {avg_cer:>7.1%}")
    print()
    print(f"结论: 在 {len(results)} 条长音频上，平均 CER = {avg_cer:.1%}")
else:
    print("没有可用的测试结果")

## 6. 保存结果与网页报告

In [ ]:
## 7. 与基线对比（可选）

用未微调的原始 SenseVoice 跑同样的测试，对比 CER 差异。

## 6. 与基线对比（可选）

用未微调的原始 SenseVoice 跑同样的测试，对比 CER 差异。

In [ ]:
# 取消注释以运行基线对比
# baseline_model = AutoModel(
#     model="iic/SenseVoiceSmall",
#     hub="ms",
#     vad_model="iic/speech_fsmn_vad_zh-cn-16k-common-pytorch",
#     vad_kwargs={"max_single_segment_time": 30000},
# )
#
# for s in samples[:2]:
#     if not os.path.exists(s['audio']):
#         continue
#     result = baseline_model.generate(input=s['audio'], language="auto", use_itn=True)
#     pred = "".join(r['text'] for r in result if 'text' in r)
#     cer = compute_cer(pred, s['text'])
#     print(f"{s['speaker']} | 基线CER: {cer:.1%}")